In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import utils
import seaborn as sns
from pathlib import Path

path_figs = "./Figs/"

fingers = utils.fingers
index_finger = utils.index_finger

iti = utils.iti
planTime = utils.planTime
feedbackTime = utils.feedbackTime

total_sub_num = utils.total_sub_num
num_blocks = utils.num_blocks
conditions = utils.conditions

# Fill in after collection, or leave None to use every .dat file found
sub_nums = None

utils.set_figure_style("1col")
sns.color_palette('colorblind')


[(0.00392156862745098, 0.45098039215686275, 0.6980392156862745),
 (0.8705882352941177, 0.5607843137254902, 0.0196078431372549),
 (0.00784313725490196, 0.6196078431372549, 0.45098039215686275),
 (0.8352941176470589, 0.3686274509803922, 0.0),
 (0.8, 0.47058823529411764, 0.7372549019607844),
 (0.792156862745098, 0.5686274509803921, 0.3803921568627451),
 (0.984313725490196, 0.6862745098039216, 0.8941176470588236),
 (0.5803921568627451, 0.5803921568627451, 0.5803921568627451),
 (0.9254901960784314, 0.8823529411764706, 0.2),
 (0.33725490196078434, 0.7058823529411765, 0.9137254901960784)]

In [2]:
# reload utils
import importlib
importlib.reload(utils)


<module 'utils' from '/Users/amin/projects/LearningDynamics/ChordAdaptationDynamics/SingleFingerGainGeneralizationV2/utils.py'>

Copy `.dat` files into `Single_Finger_Gain_Generalization_V2/` after collection.

Expected names: `single_gain_generalization_v2_<subNum>.dat`.

Each subject has 12 blocks (4 cycles × washout / adaptation / test).


In [3]:
available = utils.list_dat_subjects()
print("Found .dat subjects:", available)

if sub_nums is None:
    sub_nums = available

if not sub_nums:
    raise FileNotFoundError(
        f"No .dat files in {utils.dat_dir().resolve()}. "
        "After collection, copy single_gain_generalization_v2_<subNum>.dat "
        "into that folder and re-run."
    )

missing = [s for s in sub_nums if s not in available]
if missing:
    raise FileNotFoundError(f"Missing .dat files for subjects: {missing}")

subjs_list = utils.read_dat_files_subjs_list(sub_nums)
subjs = pd.concat(subjs_list, ignore_index=True)

subjs['chord'] = subjs.apply(utils.extract_chord_form_target_force, axis=1)
subjs['condition'] = subjs.apply(utils.extract_condition, axis=1)
subjs['num_targets'] = subjs[[f'targetForce{i}' for i in range(1, 6)]].ne(0).sum(axis=1)
subjs['count'] = 1

subjs = utils.add_block_type(subjs)
subjs = utils.add_adaptation_condition(subjs)
subjs = utils.add_force_errors(subjs)
subjs = utils.add_total_trial_num(subjs)
subjs = utils.add_day(subjs)
subjs = utils.add_trial_num_within_block(subjs)
subjs = utils.add_trial_num_within_condition_within_block(subjs)
subjs.reset_index(drop=True, inplace=True)


Found .dat subjects: [1, 2, 3, 4, 5]


In [4]:
subjs

,BN,TN,subNum,targetForce1,targetForce2,targetForce3,targetForce4,targetForce5,endForce1,endForce2,...,producedForce4,expectedForce4,forceError4,producedForce5,expectedForce5,forceError5,TotalTrialNum,day,trial_num_within_block,trial_num_within_condition
0,1,1,1,0.0,4.0,0.0,0.0,0.0,-0.168,2.691,...,-0.240,0.0,-0.240,-0.087,0.0,-0.087,1,1,1,1
1,1,2,1,0.0,2.0,0.0,0.0,0.0,-0.032,2.305,...,-0.107,0.0,-0.107,-0.045,0.0,-0.045,2,1,2,1
2,1,3,1,0.0,-4.0,0.0,0.0,0.0,-0.053,-5.095,...,0.008,0.0,0.008,-0.065,0.0,-0.065,3,1,3,1
3,1,4,1,0.0,-4.0,0.0,0.0,0.0,-0.029,-4.847,...,-0.060,0.0,-0.060,-0.052,0.0,-0.052,4,1,4,2
4,1,5,1,0.0,-2.0,0.0,0.0,0.0,-0.000,-1.389,...,-0.048,0.0,-0.048,-0.011,0.0,-0.011,5,1,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2795,12,56,5,0.0,-4.0,0.0,0.0,0.0,-0.038,-9.138,...,-0.045,0.0,-0.045,0.025,0.0,0.025,556,1,56,14
2796,12,57,5,0.0,4.0,0.0,0.0,0.0,-2.072,3.425,...,-1.363,0.0,-1.363,-0.177,0.0,-0.177,557,1,57,14
2797,12,58,5,0.0,4.0,0.0,0.0,0.0,-0.719,2.650,...,-1.252,0.0,-1.252,-0.067,0.0,-0.067,558,1,58,15
2798,12,59,5,0.0,2.0,0.0,0.0,0.0,-0.067,1.762,...,-0.036,0.0,-0.036,-0.008,0.0,-0.008,559,1,59,15


In [5]:
subjs.columns


Index(['BN', 'TN', 'subNum', 'targetForce1', 'targetForce2', 'targetForce3',
       'targetForce4', 'targetForce5', 'endForce1', 'endForce2', 'endForce3',
       'endForce4', 'endForce5', 'isTargetVisible', 'gain', 'planTime',
       'feedbackTime', 'iti', 'forceGain', 'trialCorr', 'trialErrorType',
       'chord', 'condition', 'num_targets', 'count', 'block_type', 'cycle',
       'adaptation_condition', 'producedForce1', 'expectedForce1',
       'forceError1', 'producedForce2', 'expectedForce2', 'forceError2',
       'producedForce3', 'expectedForce3', 'forceError3', 'producedForce4',
       'expectedForce4', 'forceError4', 'producedForce5', 'expectedForce5',
       'forceError5', 'TotalTrialNum', 'day', 'trial_num_within_block',
       'trial_num_within_condition'],
      dtype='str')

In [6]:
keep_cols = [
    'subNum', 'BN', 'TN', 'TotalTrialNum', 'trial_num_within_block', 'trial_num_within_condition',
    'cycle', 'block_type', 'adaptation_condition',
    'targetForce1', 'targetForce2', 'targetForce3', 'targetForce4', 'targetForce5',
    'endForce1', 'endForce2', 'endForce3', 'endForce4', 'endForce5',
    'producedForce1', 'producedForce2', 'producedForce3', 'producedForce4', 'producedForce5',
    'expectedForce1', 'expectedForce2', 'expectedForce3', 'expectedForce4', 'expectedForce5',
    'forceError1', 'forceError2', 'forceError3', 'forceError4', 'forceError5',
    'isTargetVisible', 'gain', 'forceGain', 'trialCorr', 'trialErrorType',
    'num_targets', 'chord', 'condition', 'day', 'count',
]
subjs = subjs[keep_cols]


In [7]:
subjs.head()


,subNum,BN,TN,TotalTrialNum,trial_num_within_block,trial_num_within_condition,cycle,block_type,adaptation_condition,targetForce1,...,isTargetVisible,gain,forceGain,trialCorr,trialErrorType,num_targets,chord,condition,day,count
0,1,1,1,1,1,1,1,washout,4.0,0.0,...,1,1.0,1.0,1,0,1,-^---,4.0,1,1
1,1,1,2,2,2,1,1,washout,4.0,0.0,...,1,1.0,1.0,1,0,1,-^---,2.0,1,1
2,1,1,3,3,3,1,1,washout,4.0,0.0,...,1,1.0,1.0,1,0,1,-v---,-4.0,1,1
3,1,1,4,4,4,2,1,washout,4.0,0.0,...,1,1.0,1.0,1,0,1,-v---,-4.0,1,1
4,1,1,5,5,5,1,1,washout,4.0,0.0,...,1,1.0,1.0,1,0,1,-v---,-2.0,1,1


In [8]:
print(subjs.groupby('subNum').agg(
    n_trials=('TN', 'count'),
    n_blocks=('BN', 'nunique'),
    n_correct=('trialCorr', 'sum'),
))
print(subjs['condition'].value_counts().sort_index())
print(subjs.groupby(['block_type', 'gain', 'isTargetVisible']).size())
print(subjs.groupby(['subNum', 'cycle', 'adaptation_condition', 'block_type']).size().head(20))


        n_trials  n_blocks  n_correct
subNum                               
1            560        12        521
2            560        12        513
3            560        12        529
4            560        12        553
5            560        12        505
condition
-4.0    700
-2.0    700
 2.0    700
 4.0    700
Name: count, dtype: int64
block_type  gain  isTargetVisible
adaptation  2.0   1                   800
test        1.0   0                  1200
washout     1.0   1                   800
dtype: int64
subNum  cycle  adaptation_condition  block_type
1       1       4.0                  adaptation    40
                                     test          60
                                     washout       40
        2       2.0                  adaptation    40
                                     test          60
                                     washout       40
        3      -4.0                  adaptation    40
                                     test          

In [9]:
Path(utils.path_misc).mkdir(parents=True, exist_ok=True)
subjs.to_csv(utils.path_misc + 'subjs.csv', index=False, sep='\t')
print('saved', utils.path_misc + 'subjs.csv')


saved ./Single_Finger_Gain_Generalization_V2_miscs/subjs.csv
